In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Tehnici de atenuare și suprimare a erorilor

> **Note:** Versiunea beta a unui nou model de execuție este acum disponibilă. Modelul de execuție direcționată oferă mai multă flexibilitate la personalizarea fluxului tău de lucru pentru atenuarea erorilor. Consultă ghidul [Modelul de execuție direcționată](/guides/directed-execution-model) pentru mai multe informații.


<details>
<summary><b>Versiuni de pachete</b></summary>

Codul de pe această pagină a fost dezvoltat folosind următoarele cerințe.
Recomandăm utilizarea acestor versiuni sau a unora mai noi.

```
qiskit-ibm-runtime~=0.43.1
```
</details>
Tehnicile de atenuare și suprimare a erorilor sunt utilizate pentru a îmbunătăți calitatea rezultatelor la scalarea la sarcini de lucru mai mari. Această pagină oferă explicații de nivel înalt ale tehnicilor de suprimare și atenuare a erorilor disponibile prin Qiskit Runtime.

Următoarea celulă de cod importă primitivul Estimator și creează un Backend care va fi utilizat pentru inițializarea Estimator în celulele de cod ulterioare.

In [1]:
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.least_busy()

## Decuplare dinamică
Circuitele cuantice sunt executate pe hardware-ul IBM&reg; ca secvențe de impulsuri de microunde care trebuie programate și rulate la intervale de timp precise.
Din păcate, interacțiunile nedorite dintre Qubiți pot duce la erori coerente pe Qubiții inactivi. Decuplarea dinamică funcționează prin inserarea de secvențe de impulsuri pe Qubiții inactivi pentru a anula aproximativ efectul acestor erori. Fiecare secvență de impulsuri inserată echivalează cu o operație de identitate, dar prezența fizică a impulsurilor are efectul de a suprima erorile.
Există multe posibilități de alegere a secvențelor de impulsuri, iar care secvență este mai bună pentru fiecare caz particular rămâne un [domeniu activ de cercetare](https://journals.aps.org/prapplied/abstract/10.1103/PhysRevApplied.20.064027).

Reține că decuplarea dinamică este utilă în principal pentru Circuite care conțin goluri în care unii Qubiți stau inactivi fără nicio operație care acționează asupra lor. Dacă operațiile din Circuit sunt foarte dense, astfel încât toți Qubiții sunt ocupați cea mai mare parte a timpului, atunci adăugarea de impulsuri de decuplare dinamică s-ar putea să nu îmbunătățească performanța. De fapt, ar putea chiar să înrăutățească performanța din cauza imperfecțiunilor impulsurilor în sine.

Diagrama de mai jos ilustrează decuplarea dinamică cu o secvență de impulsuri XX. Circuitul abstract din stânga este mapat pe un program de impulsuri de microunde în dreapta sus. Dreapta jos ilustrează același program, dar cu o secvență de două impulsuri X inserate în timpul unei perioade inactive a primului Qubit.

![Ilustrarea decuplării dinamice](../docs/images/guides/error-mitigation-explanation/dd.avif)

Decuplarea dinamică poate fi activată prin setarea `enable` la `True` în [opțiunile de decuplare dinamică](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-dynamical-decoupling-options). Opțiunea `sequence_type` poate fi utilizată pentru a alege dintre mai multe secvențe de impulsuri diferite. Tipul implicit de secvență este `"XX"`.

Următoarea celulă de cod arată cum să activezi decuplarea dinamică pentru Estimator și să alegi o secvență de decuplare dinamică.

In [2]:
estimator = Estimator(mode=backend)
estimator.options.dynamical_decoupling.enable = True
estimator.options.dynamical_decoupling.sequence_type = "XpXm"

## Răsucirea Pauli
Răsucirea, cunoscută și ca [compilare randomizată](https://journals.aps.org/pra/abstract/10.1103/PhysRevA.94.052325), este o tehnică larg utilizată pentru conversia canalelor arbitrare de zgomot în canale de zgomot cu o structură mai specifică.

Răsucirea Pauli este un tip special de răsucire care utilizează operații Pauli. Are efectul de a transforma orice canal cuantic într-un canal Pauli. Efectuată singură, poate atenua zgomotul coerent, deoarece zgomotul coerent tinde să se acumuleze pătratic cu numărul de operații, în timp ce zgomotul Pauli se acumulează liniar. Răsucirea Pauli este adesea combinată cu alte tehnici de atenuare a erorilor care funcționează mai bine cu zgomotul Pauli decât cu zgomotul arbitrar.

Răsucirea Pauli este implementată prin sandwicharea unui set ales de Gate-uri cu Gate-uri Pauli cu un singur Qubit alese aleatoriu, în așa fel încât efectul ideal al Gate-ului rămâne același. Rezultatul este că un singur Circuit este înlocuit cu un ansamblu aleatoriu de Circuite, toate cu același efect ideal. La eșantionarea Circuit-ului, eșantioanele sunt extrase din mai multe instanțe aleatorii, nu doar dintr-una singură.

![Ilustrarea răsucirii Pauli](../docs/images/guides/error-mitigation-explanation/pauli_twirling.avif)

Deoarece majoritatea erorilor din hardware-ul cuantic actual provin din Gate-urile cu doi Qubiți, această tehnică este adesea aplicată exclusiv Gate-urilor (native) cu doi Qubiți. Diagrama următoare ilustrează câteva răsuciri Pauli pentru Gate-urile CNOT și ECR. Fiecare Circuit dintr-un rând are același efect ideal.

![Ilustrarea răsucirilor de Gate](../docs/images/guides/error-mitigation-explanation/gate_twirls.avif)

Răsucirea Pauli poate fi activată prin setarea `enable_gates` la `True` în [opțiunile de răsucire](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options). Alte opțiuni notabile includ:

- `num_randomizations`: Numărul de instanțe de Circuit de extras din ansamblul de Circuite răsucite.
- `shots_per_randomization`: Numărul de shot-uri de eșantionat din fiecare instanță de Circuit.

Următoarea celulă de cod arată cum să activezi răsucirea Pauli și să setezi aceste opțiuni pentru Estimator. Niciuna dintre aceste opțiuni nu trebuie setată explicit.

In [3]:
estimator = Estimator(mode=backend)
estimator.options.twirling.enable_gates = True
estimator.options.twirling.num_randomizations = 32
estimator.options.twirling.shots_per_randomization = 100

## Extincția erorilor de citire prin răsucire (TREX)
[Extincția erorilor de citire prin răsucire (TREX)](https://journals.aps.org/pra/abstract/10.1103/PhysRevA.105.032620) atenuează efectul erorilor de măsurare pentru estimarea valorilor așteptate ale observabilelor Pauli.
Se bazează pe noțiunea de măsurători răsucite, care sunt realizate prin substituirea aleatorie a Gate-urilor de măsurare cu o secvență de (1) un Gate Pauli X, (2) o măsurătoare și (3) o inversare a bitului clasic. La fel ca în răsucirea standard a Gate-urilor, această secvență este echivalentă cu o măsurătoare simplă în absența zgomotului, așa cum este ilustrat în diagrama următoare:

![Ilustrarea răsucirii măsurătorii](../docs/images/guides/error-mitigation-explanation/measurement_twirling.avif)

În prezența erorii de citire, răsucirea măsurătorii are efectul de a diagonaliza matricea de transfer a erorii de citire, făcând-o ușor de inversat. Estimarea matricei de transfer a erorii de citire necesită executarea unor Circuite de calibrare suplimentare, ceea ce introduce un mic cost suplimentar.

TREX poate fi activat prin setarea `measure_mitigation` la `True` în [opțiunile de reziliență Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2) pentru Estimator. Opțiunile pentru învățarea zgomotului de măsurare sunt descrise [aici](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-measure-noise-learning-options). Ca și în cazul răsucirii Gate-urilor, poți seta numărul de randomizări ale Circuit-ului și numărul de shot-uri pe randomizare.

Următoarea celulă de cod arată cum să activezi TREX și să setezi aceste opțiuni pentru Estimator. Niciuna dintre aceste opțiuni nu trebuie setată explicit.

In [4]:
estimator = Estimator(mode=backend)
estimator.options.resilience.measure_mitigation = True
estimator.options.resilience.measure_noise_learning.num_randomizations = 32
estimator.options.resilience.measure_noise_learning.shots_per_randomization = 100

<span id="zne"></span>
## Extrapolarea la zgomot zero (ZNE)
Extrapolarea la zgomot zero (ZNE) este o tehnică pentru atenuarea erorilor în estimarea valorilor așteptate ale observabilelor. Deși îmbunătățește adesea rezultatele, nu garantează producerea unui rezultat nedeformat.

ZNE constă din două etape:

1. _Amplificarea zgomotului_: Circuit-ul cuantic original este executat de mai multe ori la diferite niveluri de zgomot.
2. _Extrapolarea_: Rezultatul ideal este estimat prin extrapolarea rezultatelor valorilor așteptate zgomotoase la limita de zgomot zero.

Atât etapele de amplificare a zgomotului, cât și cele de extrapolare pot fi implementate în multe moduri diferite. Qiskit Runtime implementează amplificarea zgomotului prin „pliere digitală a Gate-urilor", ceea ce înseamnă că Gate-urile cu doi Qubiți sunt înlocuite cu secvențe echivalente ale Gate-ului și inversul acestuia. De exemplu, înlocuirea unui unitar $U$ cu $U U^\dagger U$ ar produce un factor de amplificare a zgomotului de 3. Pentru extrapolare, poți alege dintre mai multe forme funcționale, inclusiv o ajustare liniară sau o ajustare exponențială.
Imaginea de mai jos ilustrează plierea digitală a Gate-ului în stânga și procedura de extrapolare în dreapta.

![Ilustrarea ZNE](../docs/images/guides/error-mitigation-explanation/zne.avif)

ZNE poate fi activat prin setarea `zne_mitigation` la `True` în [opțiunile de reziliență Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2) pentru Estimator.
Opțiunile Qiskit Runtime pentru ZNE sunt descrise [aici](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options). Următoarele opțiuni sunt notabile:

- `noise_factors`: Factorii de zgomot de utilizat pentru amplificarea zgomotului.
- `extrapolator`: Forma funcțională de utilizat pentru extrapolare.

Următoarea celulă de cod arată cum să activezi ZNE și să setezi aceste opțiuni pentru Estimator. Niciuna dintre aceste opțiuni nu trebuie setată explicit.

In [5]:
estimator = Estimator(mode=backend)
estimator.options.resilience.zne_mitigation = True
estimator.options.resilience.zne.noise_factors = (1, 3, 5)
estimator.options.resilience.zne.extrapolator = "exponential"

<span id="pea"></span>
## Amplificarea probabilistică a erorilor (PEA)
Una dintre principalele provocări în ZNE este amplificarea cu acuratețe a zgomotului care afectează Circuit-ul țintă. Plierea Gate-urilor oferă o modalitate ușoară de a efectua această amplificare, dar este potențial inexactă și ar putea duce la rezultate incorecte. Consultă articolul ["Scalable error mitigation for noisy quantum circuits produces competitive expectation values"](https://arxiv.org/pdf/2108.09197), și în special pagina 4 a informațiilor suplimentare pentru detalii. Amplificarea probabilistică a erorilor oferă o abordare mai precisă a amplificării erorilor prin învățarea zgomotului.

PEA este o tehnică mai sofisticată care efectuează experimente preliminare pentru a reconstrui zgomotul și apoi folosește aceste informații pentru a efectua o amplificare precisă. Începe prin învățarea modelului de zgomot răsucit al fiecărui strat de Gate-uri de înlănțuire din Circuit înainte ca acestea să fie rulate (vezi [LayerNoiseLearningOptions](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-layer-noise-learning-options) pentru opțiunile de învățare relevante). După faza de învățare, Circuitele sunt executate la fiecare factor de zgomot, unde fiecare strat de înlănțuire al Circuitelor este amplificat prin injectarea probabilistică a zgomotului cu un singur Qubit proporțional cu modelul de zgomot învățat corespunzător. Consultă articolul ["Evidence for the utility of quantum computing before fault tolerance"](https://www.nature.com/articles/s41586-023-06096-3) pentru mai multe detalii.

PEA constă din trei etape:
1. _Învățarea_: Modelul de zgomot răsucit al fiecărui strat de Gate-uri de înlănțuire din Circuit este învățat.
1. _Amplificarea zgomotului_: Circuit-ul cuantic original este executat de mai multe ori la diferiți factori de zgomot.
2. _Extrapolarea_: Rezultatul ideal este estimat prin extrapolarea rezultatelor valorilor așteptate zgomotoase la limita de zgomot zero.

Pentru experimente la scară de utilitate, PEA este adesea cea mai bună alegere.

Deoarece PEA este o tehnică de amplificare a zgomotului ZNE, trebuie să activezi și ZNE prin setarea `resilience.zne_mitigation = True`. Alte opțiuni [`resilience.zne`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options) pot fi folosite suplimentar pentru a seta extrapolatori, niveluri de amplificare și altele. PEA necesită un model de zgomot, care este generat automat la utilizarea primitivelor.

Fragmentul următor oferă un exemplu în care PEA este utilizat pentru a atenua rezultatul unui job Estimator:

In [6]:
estimator = Estimator(mode=backend)
estimator.options.resilience.zne_mitigation = True
estimator.options.resilience.zne.amplifier = "pea"

<span id="pec"></span>
## Anularea probabilistică a erorilor (PEC)
Anularea probabilistică a erorilor (PEC) este o tehnică pentru atenuarea erorilor în estimarea valorilor așteptate ale observabilelor. Spre deosebire de ZNE, returnează o estimare nedeformată a valorii așteptate. Cu toate acestea, implică în general un cost suplimentar mai mare.

În PEC, efectul unui Circuit țintă ideal este exprimat ca o combinație liniară de Circuite zgomotoase care sunt efectiv implementabile în practică:

$$
\mathcal{O}_{\text{ideal}} = \sum_{i} \eta_i \mathcal{O}_{noisy, i}
$$

Ieșirea Circuit-ului ideal poate fi reprodusă apoi prin executarea unor instanțe diferite de Circuit zgomotos extrase dintr-un ansamblu aleatoriu definit de combinația liniară. Dacă coeficienții $\eta_i$ formează o distribuție de probabilitate, aceștia pot fi utilizați direct ca probabilitățile ansamblului. În practică, unii dintre coeficienți sunt negativi, astfel că formează o distribuție de quasi-probabilitate. Pot fi utilizați în continuare pentru a defini un ansamblu aleatoriu, dar există un cost de eșantionare legat de negativitatea distribuției de quasi-probabilitate, care este caracterizat de cantitatea

$$
\gamma = \sum_{i} \lvert \eta_i \rvert \geq 1.
$$

Costul de eșantionare este un factor multiplicativ pentru numărul de shot-uri necesar pentru a estima o valoare așteptată la o precizie dată, comparativ cu numărul de shot-uri care ar fi necesare din Circuit-ul ideal. Se scalează pătratic cu $\gamma$, care la rândul său se scalează exponențial cu adâncimea Circuit-ului.

PEC poate fi activat prin setarea `pec_mitigation` la `True` în [opțiunile de reziliență Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2) pentru Estimator.
Opțiunile Qiskit Runtime pentru PEC sunt descrise [aici](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-pec-options). O limită pentru costul de eșantionare poate fi setată folosind opțiunea `max_overhead`. Reține că limitarea costului de eșantionare poate face ca precizia rezultatului să depășească precizia solicitată. Valoarea implicită a `max_overhead` este 100.

Următoarea celulă de cod arată cum să activezi PEC și să setezi opțiunea `max_overhead` pentru Estimator.

In [7]:
estimator = Estimator(mode=backend)
estimator.options.resilience.pec_mitigation = True
estimator.options.resilience.pec.max_overhead = 100

## Pași următori
- Consultă [tutorialul](/tutorials/combine-error-mitigation-techniques) despre combinarea opțiunilor de atenuare a erorilor cu primitivul Estimator.
- [Configurează atenuarea erorilor.](configure-error-mitigation)
- [Configurează suprimarea erorilor.](configure-error-suppression)
- Explorează alte [opțiuni](runtime-options-overview) pentru primitivele Qiskit Runtime.
- Decide în ce [mod de execuție](execution-modes) să rulezi job-ul tău.